In [1]:
def load_data():
    """
    Simulate loading raw data with some invalid entries.
    Returns a list of dictionaries with 'value' and 'label' keys.
    """
    data = [
        {'value': 10, 'label': 'A'},
        {'value': 25, 'label': 'B'},
        {'value': -5, 'label': 'A'},   # Negative - will be cleaned
        {'value': None, 'label': 'C'},  # None - will be cleaned
        {'value': 50, 'label': 'B'},
        {'value': 30, 'label': 'A'},
        {'value': -2, 'label': 'C'},    # Negative - will be cleaned
        {'value': 100, 'label': 'B'},
        {'value': 0, 'label': 'C'},
        {'value': 75, 'label': 'A'}
    ]
    print(f"📂 Loaded {len(data)} raw samples")
    return data

def clean_data(data):
    """
    Remove entries where 'value' is None or negative.
    Returns cleaned data list.
    """
    original_count = len(data)
    cleaned_data = [item for item in data if item['value'] is not None and item['value'] >= 0]
    removed_count = original_count - len(cleaned_data)
    print(f"🧹 Cleaned data: removed {removed_count} invalid samples (None or negative)")
    return cleaned_data

def normalise_data(data):
    """
    Min-max normalise the 'value' field in place.
    Returns the normalised data (modifies original list).
    """
    if not data:
        return data
    
    values = [item['value'] for item in data]
    min_val = min(values)
    max_val = max(values)
    
    if min_val == max_val:
        # Avoid division by zero
        for item in data:
            item['value_normalised'] = 0.5
    else:
        for item in data:
            item['value_normalised'] = (item['value'] - min_val) / (max_val - min_val)
    
    print(f"📊 Normalised data: range [{min_val}, {max_val}] → [0, 1]")
    return data

def split_data(data, test_ratio=0.2):
    """
    Split data into train and test sets.
    Returns tuple (train_data, test_data).
    """
    split_index = int(len(data) * (1 - test_ratio))
    train_data = data[:split_index]
    test_data = data[split_index:]
    
    print(f"✂️  Split data: {len(train_data)} train, {len(test_data)} test (ratio={test_ratio})")
    return train_data, test_data

def run_pipeline():
    """
    Compose all functions into a complete data pipeline.
    """
    print("\n" + "=" * 60)
    print("RUNNING DATA PREPROCESSING PIPELINE")
    print("=" * 60 + "\n")
    
    # Step 1: Load data
    raw_data = load_data()
    
    # Step 2: Clean data
    cleaned_data = clean_data(raw_data)
    
    # Step 3: Normalise data
    normalised_data = normalise_data(cleaned_data)
    
    # Step 4: Split data
    train_data, test_data = split_data(normalised_data, test_ratio=0.2)
    
    # Display results
    print("\n" + "=" * 60)
    print("PIPELINE RESULTS")
    print("=" * 60)
    
    print(f"\n📊 Final dataset sizes:")
    print(f"   Training set: {len(train_data)} samples")
    print(f"   Test set: {len(test_data)} samples")
    
    print(f"\n📝 First 3 training samples:")
    for i, sample in enumerate(train_data[:3]):
        print(f"   Sample {i+1}: {sample}")
    
    print(f"\n📝 First 3 test samples:")
    for i, sample in enumerate(test_data[:3]):
        print(f"   Sample {i+1}: {sample}")
    
    # Calculate and display statistics
    train_values = [s['value'] for s in train_data]
    test_values = [s['value'] for s in test_data]
    
    print(f"\n📈 Pipeline statistics:")
    print(f"   Training value range: [{min(train_values):.1f}, {max(train_values):.1f}]")
    print(f"   Test value range: [{min(test_values):.1f}, {max(test_values):.1f}]")
    
    return train_data, test_data

# Run the complete pipeline
train_set, test_set = run_pipeline()

# Demonstrate why function composition matters
print("\n" + "=" * 60)
print("WHY FUNCTION COMPOSITION MATTERS")
print("=" * 60)
print("\nWithout composition (hard to test, reuse, debug):")
print("  all_data = split(normalise(clean(load())))  # Unreadable!")

print("\nWith composition (modular, testable, reusable):")
print("  raw = load_data()")
print("  clean = clean_data(raw)")
print("  norm = normalise_data(clean)")
print("  train, test = split_data(norm)")
print("  ✓ Each function does ONE thing well")
print("  ✓ Can test each function independently")
print("  ✓ Can swap or modify any step without breaking others")


RUNNING DATA PREPROCESSING PIPELINE

📂 Loaded 10 raw samples
🧹 Cleaned data: removed 3 invalid samples (None or negative)
📊 Normalised data: range [0, 100] → [0, 1]
✂️  Split data: 5 train, 2 test (ratio=0.2)

PIPELINE RESULTS

📊 Final dataset sizes:
   Training set: 5 samples
   Test set: 2 samples

📝 First 3 training samples:
   Sample 1: {'value': 10, 'label': 'A', 'value_normalised': 0.1}
   Sample 2: {'value': 25, 'label': 'B', 'value_normalised': 0.25}
   Sample 3: {'value': 50, 'label': 'B', 'value_normalised': 0.5}

📝 First 3 test samples:
   Sample 1: {'value': 0, 'label': 'C', 'value_normalised': 0.0}
   Sample 2: {'value': 75, 'label': 'A', 'value_normalised': 0.75}

📈 Pipeline statistics:
   Training value range: [10.0, 100.0]
   Test value range: [0.0, 75.0]

WHY FUNCTION COMPOSITION MATTERS

Without composition (hard to test, reuse, debug):
  all_data = split(normalise(clean(load())))  # Unreadable!

With composition (modular, testable, reusable):
  raw = load_data()
  c